In [15]:
# input
pfam_region_file = "../../../_database/Pfam/Pfam-A.regions.tsv"
fasta_file = "./tmp/PF01380_site_anno_with_Q8U4D1.fasta"
anno_file = "./tmp/PF01380_mbp_site_anno.tsv"

In [ ]:
import pandas as pd

df_anno = pd.read_table(anno_file, header=None, names=["seq_id", "seq_num", "resi", "metal_resi"])
ids = set(df_anno["seq_id"])
target = "Q8U4D1"
ids = ids | set([target])

df = pd.read_table(pfam_region_file, usecols=["pfamseq_acc", "pfamA_acc", "seq_start", "seq_end"])
df = df[df["pfamseq_acc"].map(lambda x: x in ids)]
df = df[df["pfamA_acc"] == "PF01380"]

In [6]:
id2seq = dict()

from Bio import SeqIO

for r in SeqIO.parse(fasta_file, "fasta"):
    if r.id.startswith("AFDB:"):
        seq_id = r.id.split("-")[1]
    else:
        seq_id = r.id.split("|")[1]
    id2seq[seq_id] = str(r.seq)

In [7]:
df_target = df[df["pfamseq_acc"] == target]
df_target
df = df[df["pfamseq_acc"] != target]

,pfamseq_acc,pfamA_acc,seq_start,seq_end
35294097,Q8U4D1,PF01380,284,413
35294098,Q8U4D1,PF01380,452,581


In [18]:
from Bio import Align

target_seq_1 = id2seq[target][283:413]
target_seq_2 = id2seq[target][451:581]

records = []

aligner = Align.PairwiseAligner(scoring="blastp")
for _, row in df.iterrows():
    seq_id = row["pfamseq_acc"]
    frag_id = f"{seq_id}_{row['seq_start']}_{row['seq_end']}"
    frag = id2seq[seq_id][(row['seq_start'] - 1): row['seq_end']]


    aligns_1 = aligner.align(target_seq_1, frag)
    aligns_2 = aligner.align(target_seq_2, frag)
    

    result = 0
    if aligns_1.score > aligns_2.score: result = 1
    elif aligns_2.score > aligns_1.score: result = 2
    records.append({
        "seq_id": seq_id,
        "frag_id": frag_id,
        "match": result
    })

In [20]:
df_match = pd.DataFrame(records)
df_match = pd.merge(df_match, df_anno, on="seq_id")
df_match[df_match["match"] == 2]
df_match[df_match["match"] == 0]

,seq_id,frag_id,match,seq_num,resi,metal_resi
1673,V5S9S7,V5S9S7_44_174,2,79,H,ZN
2685,A0A229USX0,A0A229USX0_38_170,2,75,H,ZN
2898,M4S5Z3,M4S5Z3_42_174,2,79,H,ZN
2949,A0A5C5W1V3,A0A5C5W1V3_36_168,2,73,H,ZN
4020,A0A0Q7E5L6,A0A0Q7E5L6_37_170,2,75,H,ZN
4166,W0A707,W0A707_42_174,2,79,H,ZN


,seq_id,frag_id,match,seq_num,resi,metal_resi
3719,A0A074MBA1,A0A074MBA1_46_179,0,84,H,ZN
